The idea with any vectr database is , we need to create a Vector Store:(persist embeddings into disk)

Local development-> Chroma DB (best for local development, prototyping or when working with small datasets).
Free, not setup, pip install langchain-chroma

FAISS, best for speed, in-memory, free, for local development.
Pinecone is best for production and managed, not free, type is Cloud.
weaviate is best for self-hosted,hybrid, it is free/paid and type is hybrid.
Qdrant is best for performance, filtering, it is free/paid, type is hybrid.

when the query comes in , we specify how many similar documents we want. k=4 means return top 4 similar documents.

Vector Store Architecture

Documents(text chunks) -> embeddings (vectors) -> vector Store(indexed) -> similarity_search (query) -> relevant documents (doc1,doc2,doc3,doc4)

Key Vector Store Methods

1)Add new documents
vectorstore.add_documents(docs)

2)Find Similar
vectorstore.similarity_search(q,k=4)

3)Results+Scores
Returns (docs,scores) as tuples

4)For chains(to connect vector stores to rag chains)
vectorstore.as_retriever(search_kwargs={...})



In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import tempfile
import shutil

load_dotenv()

embeddings_model=OpenAIEmbeddings(model="text-embedding-3-small")

SAMPLE_DOCS=[
    Document(page_content="LangChain is a framework for building applications powered by LLMs by connecting models with prompts, tools, data, memory, and retrieval systems.",
    metadata={"source": "langchain_docs", "topic": "overview"},),

    Document(page_content="LangGraph is a framework for building stateful, multi-step agent workflows with LLMs.",
    metadata={"source": "langgraph_docs", "topic": "overview"},),

     Document(page_content="Vector stores are databases optimized for storing and seraching embeddings.",
    metadata={"source": "vector_guide", "topic": "database"},),

     Document(page_content="RAG combines retrieval with generation for more accurate LLM response.",
    metadata={"source": "rag_guide", "topic": "architecture"},),

    Document(page_content="Embeddings are numerical vector representations of text that capture semantic meaning.",
    metadata={"source": "embedding_guide", "topic": "embeddings"},),

    Document(page_content="Chroma is an open-source vector database designed for storing and searching embeddings.",
    metadata={"source": "Chroma_docs", "topic": "database"},),

    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.",
    metadata={"source": "faiss_docs", "topic": "database"},),

     Document(page_content="Pinecone provides a managed vector database for storing, indexing, and searching vector embeddings in production applications.",
    metadata={"source": "pinecone_docs", "topic": "database"},),

]

def chroma_basics():
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore=Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=tmpdir,
        )

        print(f"vector store created {vectorstore._collection.count()} and persisted.")

        #perform similarity search
        query = "What is LangChain?"

        results = vectorstore.similarity_search(query, k = 2)

        print(f"Top 2 results for query '{query}':")

        for i, doc in enumerate(results):
            print(f"Result {i+1}: {doc.page_content} (Source: {doc.metadata['source']})")


def demo_similarity_search_with_score():
    #tempfile is a built-in Python module that creates temporary files and directories.
    #gives distance score, farther from 0 means lesser similar/relevant.
    # Store the created object (or its useful value) in the variable tmpdir.
    # So tmpdir is simply a variable containing the path to the temporary directory.

    #with creates a context.It means:Create or open something.Execute the code inside the block..When finished, automatically clean it up
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore=Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=tmpdir,
        )
        #perform similarity search
        query = "What kind of databases are there?"

        results_with_scores = vectorstore.similarity_search_with_score(query, k = 3)

        print(f"Top 3 results for query '{query}':")

        for i,(doc,score) in enumerate(results_with_scores):
            print(f"Result {i+1} : {doc.page_content} Score: {score:.4f} Source: {doc.metadata['source']}")

def demo_metadata_filtering():

    with tempfile.TemporaryDirectory() as tmpdir:
     # create vector store from documents
        vectorstore=Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=tmpdir,
        )
        #perform similarity search
        query = "What is LangChain?"

        filter_criteria={"topic":"database"}
        filtered_results = vectorstore.similarity_search(query, k = 3, filter=filter_criteria)
        
        print(f"\nResults with metadata filtering for query '{query}':")

        for i, doc in enumerate (filtered_results):
            print(f"Result {1+1}: {doc.page_content} Source: {doc.metadata['source']}")

if __name__=="__main__":
    # chroma_basics()
    #demo_similarity_search_with_score()
    demo_metadata_filtering()


#ERROR RUNNING ABOVE CELL
The crucial thing is that Windows doesn't allow a file to be deleted while another process/object still has it open.

But Chroma uses:

chroma.sqlite3

and the Chroma client/vector store may still have an open SQLite connection when Python reaches Delete this temporary directory.

Windows says:

"No. chroma.sqlite3 is still open."

Your Chroma SQLite database is still open when TemporaryDirectory tries to delete it, and Windows prevents deletion of a file that's currently being used.

Important: Python Version for ChromaDB (Use 3.11 or 3.12)
If you are using Python 3.14 and run into issues with ChromaDB, this is expected right now. Python 3.14 is very new and most libraries have not fully caught up yet. ChromaDB relies on Pydantic for configuration, and the version it uses internally (Pydantic v1) has known compatibility issues with Python 3.14's updated type inference and typing internals.

Recommended solution: use Python 3.11 or 3.12. These versions are currently the safest choices for ChromaDB and most AI/ML libraries.

Using pyenv:

pyenv install 3.12.2
pyenv local 3.12.2
Using uv or conda:

uv venv --python 3.12
or create a new conda env with Python 3.12


Why this happens:

Python 3.14 introduced changes to how types are handled internally, and many libraries (including Pydantic v1 and projects built on top of it like ChromaDB) need time to update and release compatible versions. ChromaDB will likely support Python 3.14 in a future release once those updates are in place.



Pro tip: For production-grade AI/ML projects, stick with Python 3.11 or 3.12 for now. They are stable, fast, and have the best ecosystem and library compatibility. If you are on 3.14 and things are breaking, do not debug for hours - just switch your environment to Python 3.11 or 3.12 and rerun the Chroma sections.

similarity score where: Larger value = more similar, Smaller value = less similar.

Similarity score= 1 / (1 + distance)



In [ ]:
#METADATA FILTERING

